# Inclui+ Vision: treinamento e avaliação

Passos 3 e 4 do projeto, para rodar no **Google Colab**.

Antes de começar, ative a GPU: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

O notebook só orquestra. A lógica fica nos scripts de `src/`, que também rodam no computador:

| Etapa | O que acontece | Script |
|---|---|---|
| 1 a 3 | Ambiente, código e imagens | |
| 4 | Une as fontes, unifica as classes e divide em 70/20/10 | `src/preparar_dataset.py` |
| 5 | Mostra imagens com as anotações, para conferir o mapa de classes | `src/ver_amostras.py` |
| 6 | Fine-tuning do YOLO11 a partir dos pesos do COCO | (aqui) |
| 7 | Métricas no conjunto de teste, matriz de confusão e exemplos | `src/avaliar.py` |
| 8 | Experimentos do relatório: `yolo11n` × `yolo11s` e com × sem fotos próprias | |
| 9 | Guarda os pesos e os resultados no Drive | |

## 1. Configuração e ambiente

In [ ]:
# Ajuste estes valores e rode as células na ordem.
REPO_URL = ""       # ex.: "https://github.com/seu-usuario/inclui-vision.git". Vazio = enviar o .zip do repositório
PASTA_DRIVE = "/content/drive/MyDrive/inclui-vision"   # .zip dos datasets e treinos salvos
MODELO_BASE = "yolo11n.pt"   # "n" = nano (mais rápido); "s" = small (mais preciso)
EPOCAS = 100
IMGSZ = 640
BATCH = 16

In [ ]:
%pip install -q "ultralytics>=8.3.0" albumentations

import os, shutil, subprocess, zipfile
from pathlib import Path

import torch
import ultralytics
from IPython.display import Image, display

ultralytics.checks()
assert torch.cuda.is_available(), (
    "Sem GPU. Ative em Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4."
)

In [ ]:
# O Drive guarda os .zip dos datasets e os treinos, para nada se perder se o Colab desconectar.
from google.colab import drive
drive.mount("/content/drive")

Path(PASTA_DRIVE, "brutos").mkdir(parents=True, exist_ok=True)
Path(PASTA_DRIVE, "runs").mkdir(parents=True, exist_ok=True)

## 2. Código do projeto

In [ ]:
RAIZ = Path("/content/inclui-vision")

if not RAIZ.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(RAIZ)], check=True)
    else:
        from google.colab import files
        enviado = files.upload()          # escolha o inclui-vision.zip
        with zipfile.ZipFile(next(iter(enviado))) as z:
            z.extractall("/content")

assert (RAIZ / "src" / "preparar_dataset.py").exists(), "Não achei o repositório em /content/inclui-vision."
os.chdir(RAIZ)
for pasta in ("results", "weights"):
    Path(pasta).mkdir(exist_ok=True)
print("Pasta de trabalho:", Path.cwd())

## 3. Imagens

Coloque em `MyDrive/inclui-vision/brutos/` um `.zip` por fonte, exportado no formato **YOLOv11** (ou YOLOv8, que é igual em disco):

- datasets públicos do Roboflow Universe (botão *Download Dataset*);
- as fotos do grupo, anotadas no Roboflow ou no CVAT;
- opcionalmente, um `.zip` só com fotos sem nenhum recurso de acessibilidade (`negativas.zip`).

O nome de cada `.zip` deve ser o da `pasta` correspondente em `dataset/fontes.yaml` (por exemplo, `stairs-handrail.zip`). Deixe em `fontes.yaml` só as fontes que você realmente baixou.

Se a versão do Roboflow já tiver *augmentation*, gere uma versão sem: o treino abaixo já aplica augmentation, e só no conjunto de treino.

In [ ]:
BRUTOS = RAIZ / "dataset" / "brutos"
BRUTOS.mkdir(parents=True, exist_ok=True)

zips = sorted(Path(PASTA_DRIVE, "brutos").glob("*.zip"))
assert zips, f"Nenhum .zip em {PASTA_DRIVE}/brutos."

for arquivo in zips:
    destino = BRUTOS / arquivo.stem
    if not destino.exists():
        with zipfile.ZipFile(arquivo) as z:
            z.extractall(destino)
    n = sum(1 for p in destino.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    print(f"{arquivo.stem}: {n} imagens")

In [ ]:
# Alternativa ao .zip: baixar direto do Roboflow com a sua chave de API (Settings > API Keys).
# A chave é digitada na hora e não fica salva no notebook.
#
# %pip install -q roboflow
# from getpass import getpass
# from roboflow import Roboflow
# rf = Roboflow(api_key=getpass("Chave de API do Roboflow: "))
# rf.workspace("WORKSPACE").project("PROJETO").version(1).download(
#     "yolov8", location=str(BRUTOS / "nome-da-fonte"))

## 4. Preparar o dataset

Lê `dataset/fontes.yaml`, traduz as classes de cada fonte para as do projeto, divide em **70% treino, 20% validação e 10% teste** e grava `dataset/data.yaml`.

Leia os avisos no fim da saída: eles dizem quais classes estão com poucas imagens. A tabela também fica em `dataset/resumo.md`, pronta para o relatório.

In [ ]:
!python src/preparar_dataset.py --config dataset/fontes.yaml --saida dataset --limpar

## 5. Conferir as anotações

Olhe a grade antes de treinar. Se uma rampa aparecer marcada como escada, o mapa de classes em `fontes.yaml` está errado, e é melhor descobrir agora.

In [ ]:
!python src/ver_amostras.py --n 12 --saida results/amostras_treino.jpg
display(Image("results/amostras_treino.jpg", width=1100))

## 6. Treinamento

*Fine-tuning*: o modelo parte dos pesos pré-treinados no COCO e aprende as classes novas.

A *augmentation* é aplicada pelo Ultralytics durante o treino, e só nele: flip horizontal, variação de brilho e rotação de até ±15°, além das transformações padrão da biblioteca (mosaic, escala, translação, matiz e saturação). Com o pacote `albumentations` instalado, entram também, com baixa probabilidade, blur leve e ajuste de contraste.

Os treinos são salvos em `MyDrive/inclui-vision/runs/`.

In [ ]:
from ultralytics import YOLO

def treinar(nome, base=MODELO_BASE, data="dataset/data.yaml"):
    """Treina e devolve o caminho do best.pt."""
    modelo = YOLO(base)                 # pesos pré-treinados no COCO (transfer learning)
    modelo.train(
        data=str(RAIZ / data),
        epochs=EPOCAS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=20,                    # early stopping se não melhorar
        project=f"{PASTA_DRIVE}/runs",
        name=nome,
        exist_ok=True,
        seed=0,
        fliplr=0.5,                     # flip horizontal
        hsv_v=0.4,                      # brilho
        degrees=15,                     # rotação de até ±15°
    )
    return Path(modelo.trainer.best)

melhor = treinar("inclui_vision")
print("Melhores pesos:", melhor)

In [ ]:
# Se o Colab desconectar no meio do treino, monte o Drive de novo, rode as células 1 e 2
# e descomente as linhas abaixo para continuar de onde parou.
#
# from ultralytics import YOLO
# modelo = YOLO(f"{PASTA_DRIVE}/runs/inclui_vision/weights/last.pt")
# modelo.train(resume=True)
# melhor = Path(modelo.trainer.best)

In [ ]:
# Curvas de loss e de mAP por época, e os pesos finais no lugar que os scripts esperam.
pasta_treino = melhor.parents[1]
shutil.copy2(pasta_treino / "results.png", "results/results.png")
shutil.copy2(pasta_treino / "results.csv", "results/results.csv")
shutil.copy2(melhor, "weights/best.pt")
display(Image("results/results.png", width=1100))

## 7. Avaliação no conjunto de teste

O conjunto de teste não foi usado no treino nem na escolha do melhor modelo. As tabelas saem em `results/metricas.md`, prontas para o relatório.

In [ ]:
ROTULO = Path(MODELO_BASE).stem      # yolo11n
!python src/avaliar.py --pesos weights/best.pt --data dataset/data.yaml --split test --rotulo {ROTULO} --saida results
display(Image("results/confusion_matrix.png", width=750))

In [ ]:
# Prints de inferência: imagens de teste com as bounding boxes previstas.
for exemplo in sorted(Path("results/exemplos").glob("*.jpg"))[:6]:
    display(Image(str(exemplo), width=520))

## 8. Experimentos para o relatório

Os dois modelos abaixo são avaliados no **mesmo conjunto de teste** do modelo principal, e cada um ganha uma linha em `results/comparacao.csv`.

1. **Velocidade × precisão:** `yolo11n` contra `yolo11s`.
2. **Valor das fotos próprias:** treino só com os datasets públicos contra o treino completo.

In [ ]:
melhor_s = treinar("exp_yolo11s", base="yolo11s.pt")
!python src/avaliar.py --pesos "{melhor_s}" --data dataset/data.yaml --rotulo yolo11s --saida results/yolo11s

In [ ]:
if Path("dataset/data_publico.yaml").exists():
    melhor_pub = treinar("exp_so_publico", data="dataset/data_publico.yaml")
    !python src/avaliar.py --pesos "{melhor_pub}" --data dataset/data.yaml --rotulo so_publico --saida results/so_publico
else:
    print("Sem data_publico.yaml: o dataset não mistura fontes públicas e fotos próprias no treino.")

In [ ]:
import pandas as pd

comparacao = pd.read_csv("results/comparacao.csv")
display(comparacao)

## 9. Guardar os resultados

Copia tudo para `MyDrive/inclui-vision/entrega/` e baixa um `.zip`. Depois, no computador, ponha o `best.pt` em `weights/` e os arquivos de `results/` no repositório.

In [ ]:
entrega = Path(PASTA_DRIVE, "entrega")
shutil.copytree("results", entrega / "results", dirs_exist_ok=True)
(entrega / "weights").mkdir(parents=True, exist_ok=True)
shutil.copy2("weights/best.pt", entrega / "weights" / "best.pt")
shutil.copy2("dataset/resumo.md", entrega / "resumo_dataset.md")
shutil.copy2("dataset/manifesto.csv", entrega / "manifesto.csv")

shutil.make_archive("/content/inclui-vision-resultados", "zip", entrega)
from google.colab import files
files.download("/content/inclui-vision-resultados.zip")